In [11]:
import torch

print("Number of GPU: ", torch.cuda.device_count())
print("GPU Name: ", torch.cuda.get_device_name())

Number of GPU:  1
GPU Name:  NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [6]:
import tensorflow as tf

# Check if GPU is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Set memory growth to avoid memory allocation issues
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"TensorFlow is using GPU: {tf.config.experimental.get_device_details(gpus[0])['device_name']}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found for TensorFlow.")

No GPU found for TensorFlow.


In [13]:

import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Paths to datasets for each plane (folder names may be mislabeled)
base_dirs = {
    "sagittal": r"D:/processed_slices/sagittal",
    "coronal": r"D:/processed_slices/coronal",
    "axial": r"D:/processed_slices/axial"
}
# If your folders were named incorrectly (e.g. sagittal stored under an 'axial' folder),
# provide a mapping from the logical plane name to the actual folder key in `base_dirs`.
# Example from your message: true sagittal stored in 'axial', true axial stored in 'coronal',
# and true coronal stored in 'sagittal'. Adjust the mapping below if different.
plane_folder_map = {"sagittal": "axial", "axial": "coronal", "coronal": "sagittal"}
# Build corrected mapping used for training: logical_plane -> actual directory path
corrected_base_dirs = {plane: base_dirs[plane_folder_map.get(plane, plane)] for plane in base_dirs}

# Image size and other parameters
IMAGE_SIZE = (176, 176)
BATCH_SIZE = 32
NUM_CLASSES = 3
EPOCHS = 10

# Function to train the model for a specific plane
def train_on_plane(plane_name, base_dir):
    print(f"\nTraining on {plane_name} plane...\n")

    # Data generators
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=10,
        zoom_range=0.1,
        horizontal_flip=True
    )
    val_test_datagen = ImageDataGenerator(rescale=1./255)

    # Load training, validation, and test data
    train_generator = train_datagen.flow_from_directory(
        os.path.join(base_dir, 'train'),
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    val_generator = val_test_datagen.flow_from_directory(
        os.path.join(base_dir, 'val'),
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    test_generator = val_test_datagen.flow_from_directory(
        os.path.join(base_dir, 'test'),
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    # Load VGG16 without top layers
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(176, 176, 3))
    for layer in base_model.layers:
        layer.trainable = False

    # Add custom classification head
    x = base_model.output
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(NUM_CLASSES, activation='softmax')(x)

    # Create and compile the model
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    # Early stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Train the model
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=[early_stop]
    )

    # Evaluate the model
    loss, acc = model.evaluate(test_generator)
    print(f"\nTest Accuracy on {plane_name} plane: {acc:.2f}")

    # Plot accuracy and loss
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(f'{plane_name.capitalize()} Plane - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{plane_name.capitalize()} Plane - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Train on each plane using the corrected mapping
for plane, base_dir in corrected_base_dirs.items():
    train_on_plane(plane, base_dir)


In [2]:
import tensorflow as tf
print("Built with GPU support:", tf.test.is_built_with_cuda())
print("GPU available:", tf.config.list_physical_devices('GPU'))

Built with GPU support: False
GPU available: []


In [4]:
import torch

print("Number of GPU: ", torch.cuda.device_count())
print("GPU Name: ", torch.cuda.get_device_name())

Number of GPU:  1
GPU Name:  NVIDIA GeForce RTX 4050 Laptop GPU


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parameters
DATA_DIR = "D:/processed_slices"
BATCH_SIZE = 32
EPOCHS = 10
NUM_CLASSES = 3
PLANES = ['axial', 'coronal', 'sagittal']

# If folders are mislabeled, map logical plane -> actual folder name
# Example mapping based on your message: true sagittal in 'axial', true axial in 'coronal', true coronal in 'sagittal'
plane_folder_map = {'sagittal':'axial','axial':'coronal','coronal':'sagittal'}

# Transform for pretrained models
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet
        std=[0.229, 0.224, 0.225]
    )
])

# Custom ImageFolder (no additional filtering needed when we point to plane-specific folders)
class PlaneFilteredDataset(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)

# DataLoader function
def get_loader(subset, plane):
    # Map logical plane name to the actual folder containing that plane's images
    actual_plane_folder = plane_folder_map.get(plane, plane)
    folder = os.path.join(DATA_DIR, actual_plane_folder, subset)
    dataset = PlaneFilteredDataset(folder, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(subset == "train"))
    return loader

# Get model
def get_model():
    model = models.efficientnet_b0(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    return model.to(device)

# Training loop
def train(model, train_loader, val_loader, plane):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(EPOCHS):
        model.train()
        total, correct, running_loss = 0, 0, 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        val_acc = evaluate(model, val_loader)
        print(f"[{plane.upper()}] Epoch {epoch+1}/{EPOCHS} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    torch.save(model.state_dict(), f"model_{plane}.pth")
    print(f"Saved model_{plane}.pth")

# Evaluation
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

# Run training per plane
for plane in PLANES:
    print(f"\n🔁 Training for {plane.upper()} plane...\n")
    train_loader = get_loader("train", plane)
    val_loader = get_loader("val", plane)
    model = get_model()
    train(model, train_loader, val_loader, plane)



🔁 Training for AXIAL plane...



d:\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[AXIAL] Epoch 1/10 | Train Acc: 81.95% | Val Acc: 71.44%
[AXIAL] Epoch 2/10 | Train Acc: 97.24% | Val Acc: 73.68%
[AXIAL] Epoch 3/10 | Train Acc: 98.47% | Val Acc: 74.69%
[AXIAL] Epoch 4/10 | Train Acc: 98.92% | Val Acc: 76.05%
[AXIAL] Epoch 5/10 | Train Acc: 99.09% | Val Acc: 76.21%
[AXIAL] Epoch 6/10 | Train Acc: 99.33% | Val Acc: 77.27%
[AXIAL] Epoch 7/10 | Train Acc: 99.35% | Val Acc: 80.76%
[AXIAL] Epoch 8/10 | Train Acc: 99.46% | Val Acc: 76.82%
[AXIAL] Epoch 9/10 | Train Acc: 99.55% | Val Acc: 79.18%
[AXIAL] Epoch 10/10 | Train Acc: 99.61% | Val Acc: 79.30%
Saved model_axial.pth

🔁 Training for CORONAL plane...

[CORONAL] Epoch 1/10 | Train Acc: 74.56% | Val Acc: 75.86%
[CORONAL] Epoch 2/10 | Train Acc: 94.15% | Val Acc: 77.50%
[CORONAL] Epoch 3/10 | Train Acc: 96.99% | Val Acc: 78.69%
[CORONAL] Epoch 4/10 | Train Acc: 97.89% | Val Acc: 79.15%
[CORONAL] Epoch 5/10 | Train Acc: 98.39% | Val Acc: 81.29%
[CORONAL] Epoch 6/10 | Train Acc: 98.70% | Val Acc: 81.45%
[CORONAL] Epoch 7/1